# Female zebra finch HVC single-neuron song-response exercise
### Reference solution

The complete worked solution: one neuron's response to **Unfamiliar Song**, as four stacked panels.

1. **Spectrogram** of the two song presentations
2. **Oscillogram** — the song waveform, colored by the neuron's firing rate
3. **PSTH** — firing rate in 100 ms bins
4. **Raster** — one row per trial (20 trials)

Each trial is a "tandem pair": the same 3 s Unfamiliar song played twice, with a gap in between. The
display axis is `3 s pre + 3 s song A + 3 s gap + 3 s song B + 3 s post = 15 s`.

The more difficult steps (loading raw spikes, aligning to stimulus triggers, parsing the stimulus log,
detecting tandem pairs, and time-warping) have **already been done for you** — the spike times in
`data/unfamiliar_spikes.csv` are per-trial times already placed on that 15 s axis. 

**Your task:**

1. **Read the per-trial spike times**
2. **Bin them into a PSTH**
3. **Read the song WAV, plot the spectrogram and the firing-rate-colored waveform**
4. **Lay out the four panels in one figure**

Run the cells in order — **Shift + Enter** on each, or *Runtime → Run all*.

## Setup

This cell makes sure the data files are there. If you're in Colab and the `data/` folder hasn't been
downloaded yet, it accesses the repository first.

In [ ]:
# Make the data available, whether this is Colab or a local Jupyter Notebook.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/a-savoy/hvc-song-response-exercise.git"

if not Path("data/unfamiliar_spikes.csv").exists():
    print("data/ not found here — downloading the repository...")
    subprocess.run(["git", "clone", "--quiet", REPO_URL], check=True)
    os.chdir("hvc-song-response-exercise")

DATA = Path("data")
SPIKES_CSV = DATA / "unfamiliar_spikes.csv"
WAV_PATH = DATA / "unfamiliar_song.wav"
OUT_PATH = Path("my_figure.png")

print("working directory:", Path.cwd())
print("found spikes:", SPIKES_CSV.exists(), "| found wav:", WAV_PATH.exists())

Colab already has everything we need — `numpy`, `pandas`, `matplotlib`, `scipy`. There's nothing to install.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.io import wavfile
from scipy.signal import spectrogram

import warnings
warnings.filterwarnings("ignore")

## Display constants

The spikes are already on the 15 s axis, so these numbers describe the **display** — they
aren't doing any further processing to the data.

In [ ]:
PRE_WIN = 3.0        # silence before song A
STIM_DUR = 3.0       # duration of each song presentation
GAP_DISPLAY = 3.0    # gap between the two presentations
POST_WIN = 3.0       # silence after song B
BIN_DUR_MS = 100     # PSTH bin width

TOTAL_DUR = PRE_WIN + STIM_DUR + GAP_DISPLAY + STIM_DUR + POST_WIN  # 15 s

# Firing-rate color scale. In the original 4-category figure, the coolwarm colors and the PSTH y-limit
# use a scale shared across all four song types; these two numbers are that shared scale, so this
# isolated panel matches the original exactly.
# (Set both to None to auto-scale to just this condition instead.)
FR_MIN_GLOBAL = 0.0
FR_MAX_GLOBAL = 59.5

## Step 1 — read the per-trial spike times

The CSV is in "long" format: one row per spike, with the trial it belongs to. We want it as a list of
20 arrays, one per trial, because that's the shape the raster needs.

In [ ]:
def load_trials(csv_path):
    """Read the long-format CSV into a list of per-trial spike-time arrays.

    Columns: trial, spike_time_s. One row per spike; an empty trial is stored as a single NaN row so
    the trial count (and blank raster rows) are preserved. Times are already on the 15 s display axis.
    """
    df = pd.read_csv(csv_path)
    n_trials = int(df["trial"].max()) + 1
    return [df.loc[df["trial"] == t, "spike_time_s"].dropna().to_numpy() for t in range(n_trials)]


per_trial = load_trials(SPIKES_CSV)
print(f"{len(per_trial)} trials, {sum(len(s) for s in per_trial)} spikes")

## Step 2 — bin the spikes into a PSTH

Pool every spike from every trial into 100 ms bins, then divide by (bin width × number of trials) to
turn counts into an average firing rate in Hz.

In [ ]:
def compute_fr(per_trial_spikes, n_trials, total_dur=TOTAL_DUR, bin_dur_ms=BIN_DUR_MS):
    """Firing rate (Hz) per time bin, averaged across trials."""
    bins = int(total_dur * 1000 / bin_dur_ms)
    bin_edges = np.linspace(0, total_dur, bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    all_spikes = np.concatenate(per_trial_spikes) if per_trial_spikes else np.array([])
    counts, _ = np.histogram(all_spikes, bins=bin_edges)
    fr = counts / (bin_dur_ms / 1000.0 * max(1, n_trials))
    return fr, bin_edges, bin_centers


fr, bin_edges, bin_centers = compute_fr(per_trial, len(per_trial))
print(f"{len(fr)} bins of {BIN_DUR_MS} ms | peak {fr.max():.1f} Hz | mean {fr.mean():.1f} Hz")

## Steps 3 & 4 — draw the four panels

One function does all four panels. It puts the song into the display axis (song A at 3–6 s, song B at
9–12 s), then draws the spectrogram, the firing-rate-colored waveform, the PSTH bars, and the raster.

In [ ]:
def plot_cell(ax_block, song_label, wav_path, per_trial_spikes, n_trials,
              bin_dur_ms=BIN_DUR_MS, fr_min_global=None, fr_max_global=None):
    """Draw the four stacked panels (spectrogram / oscillogram / PSTH / raster)."""
    ax_spec, ax_wav, ax_bar, ax_rast = ax_block

    # --- load the song and lay it into the 15 s display (song A at 3-6 s, song B at 9-12 s) ---
    if wav_path and Path(wav_path).exists():
        wav_sr, wav = wavfile.read(wav_path)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        wav = wav.astype(np.float32)
    else:
        wav_sr, wav = 44100, np.zeros(int(STIM_DUR * 44100), dtype=np.float32)
    n_stim = int(STIM_DUR * wav_sr)
    wav = wav[:n_stim] if len(wav) >= n_stim else np.pad(wav, (0, n_stim - len(wav)))
    combined = np.concatenate([np.zeros(int(PRE_WIN * wav_sr)), wav,
                               np.zeros(int(GAP_DISPLAY * wav_sr)), wav,
                               np.zeros(int(POST_WIN * wav_sr))])
    t_combined = np.linspace(0, TOTAL_DUR, len(combined))
    song_windows = [(PRE_WIN, PRE_WIN + STIM_DUR),
                    (PRE_WIN + STIM_DUR + GAP_DISPLAY, PRE_WIN + STIM_DUR + GAP_DISPLAY + STIM_DUR)]

    # --- panel 1: spectrogram (over the two song windows only) ---
    f, t_spec, Sxx = spectrogram(combined, wav_sr, nperseg=1024, noverlap=512)
    Sxx_db = 10 * np.log10(Sxx + 1e-10)
    ax_spec.set_facecolor("black")
    for s, e in song_windows:
        idx = (t_spec >= s) & (t_spec <= e)
        if idx.any():
            ax_spec.pcolormesh(t_spec[idx], f, Sxx_db[:, idx], shading="gouraud", cmap="gray")
    ax_spec.set_xlim(0, TOTAL_DUR); ax_spec.set_ylim(500, 6500)
    ax_spec.set_xticks([]); ax_spec.set_yticks([1000, 3000, 5000])
    ax_spec.set_yticklabels(["1", "3", "5"], fontsize=6); ax_spec.set_ylabel("kHz", fontsize=7)
    ax_spec.text(0.15, 6100, song_label, ha="left", va="top", fontsize=9, fontweight="bold", color="white")

    # --- firing rate + shared coolwarm color scale ---
    fr, bin_edges, bin_centers = compute_fr(per_trial_spikes, n_trials, bin_dur_ms=bin_dur_ms)
    vmin = fr_min_global if fr_min_global is not None else fr.min()
    vmax = fr_max_global if fr_max_global is not None else fr.max()
    cnorm = Normalize(vmin=vmin, vmax=vmax)
    fr_color = lambda v: plt.cm.coolwarm(cnorm(v))

    # --- panel 2: oscillogram (waveform colored by firing rate) ---
    for i in range(len(fr)):
        bs, be = bin_edges[i], bin_edges[i + 1]
        if not any((bs < pe) and (be > ps) for ps, pe in song_windows):
            continue
        seg = (t_combined >= bs) & (t_combined < be)
        if seg.any():
            ax_wav.plot(t_combined[seg], combined[seg], color=fr_color(fr[i]), linewidth=0.6)
    ax_wav.set_xlim(0, TOTAL_DUR); ax_wav.set_xticks([]); ax_wav.set_yticks([])
    ax_wav.set_ylabel("amp", fontsize=7)

    # --- panel 3: PSTH bars ---
    ax_bar.bar(bin_centers, fr, width=bin_dur_ms / 1000, color=[fr_color(v) for v in fr], edgecolor="none")
    ax_bar.set_xlim(0, TOTAL_DUR); ax_bar.set_xticks([])
    bar_top = max(1.0, vmax * 1.05)
    ax_bar.set_ylim(0, bar_top)
    fr_ticks = [0, round(bar_top / 2), round(bar_top)]
    ax_bar.set_yticks(fr_ticks); ax_bar.set_yticklabels([str(t) for t in fr_ticks], fontsize=6)
    ax_bar.set_ylabel("Hz", fontsize=7)

    # --- panel 4: raster ---
    for ti, spikes in enumerate(per_trial_spikes):
        if len(spikes):
            ax_rast.vlines(spikes, ti + 0.58, ti + 1.42, color="black", linewidth=0.5)
    ax_rast.set_xlim(0, TOTAL_DUR); ax_rast.set_ylim(n_trials + 0.5, 0.5)
    rast_ticks = [1, max(1, n_trials // 2), n_trials]
    ax_rast.set_yticks(rast_ticks); ax_rast.set_yticklabels([str(t) for t in rast_ticks], fontsize=6)
    ax_rast.set_ylabel("trial", fontsize=7)
    ax_rast.set_xticks(np.arange(0, TOTAL_DUR + 1e-3, 3))
    ax_rast.set_xticklabels([f"{int(t)}" for t in np.arange(0, TOTAL_DUR + 1e-3, 3)], fontsize=6)
    ax_rast.set_xlabel("Time (s)", fontsize=7)

    # --- dashed stim-boundary guides + thin black frames ---
    for v in (PRE_WIN, PRE_WIN + STIM_DUR, PRE_WIN + STIM_DUR + GAP_DISPLAY,
              PRE_WIN + STIM_DUR + GAP_DISPLAY + STIM_DUR):
        for ax in ax_block:
            ax.axvline(v, color="gray", linestyle=(0, (3, 3)), linewidth=0.6, alpha=0.7)
    for ax in ax_block:
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_linewidth(0.6); sp.set_color("black")

## Build the figure

In [ ]:
fig = plt.figure(figsize=(10, 10))  # square, ~2x wider than the tall default
gs = fig.add_gridspec(4, 1, height_ratios=[1.6, 1.0, 1.2, 4.0], hspace=0.05)
axes = [fig.add_subplot(gs[k]) for k in range(4)]

plot_cell(axes, "Unfamiliar Song", WAV_PATH, per_trial, len(per_trial),
          fr_min_global=FR_MIN_GLOBAL, fr_max_global=FR_MAX_GLOBAL)

fig.suptitle("Bird_2  Exp1  SU_33", fontsize=11, fontweight="bold", y=0.93)
fig.savefig(OUT_PATH, dpi=600, bbox_inches="tight", facecolor="white")
print(f"saved: {OUT_PATH.resolve()}")

plt.show()

## Self-check

Does our figure match the provided target exactly? This compares the two PNGs pixel by pixel.

In [ ]:
import matplotlib.image as mpimg

ours = mpimg.imread(OUT_PATH)
target = mpimg.imread("target_figure.png")

if ours.shape != target.shape:
    print(f"different sizes: {ours.shape} vs {target.shape}")
else:
    diff = np.abs(ours - target)
    n_diff = int((diff.max(axis=2) > 0).sum())
    total = ours.shape[0] * ours.shape[1]
    print(f"size: {ours.shape[1]} x {ours.shape[0]} px")
    print(f"largest pixel difference: {diff.max()}")
    print(f"differing pixels: {n_diff:,} of {total:,}")
    print("\nIdentical to the target." if n_diff == 0 else "\nClose, but not pixel-identical.")